In [ ]:
#This script loads and verifies the test dataset for the GeoGuessr project
import pandas as pd
from pathlib import Path
image_dir = Path("datasets/OpenWorld/extracted_test")

#Verifies the shape
csv_path = "datasets/OpenWorld/test.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())

#This confirms data loaded as expected
print("Metadata rows:", len(df))
print("Metadata columns:", len(df.columns))

#nunique is num of unique elements
print("Unique countries:", df["country"].nunique())

print("Country counts:")
print(df["country"].value_counts().head())

image_files = [
    p for p in image_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in {
        ".jpg"
    }
]

print("Extracted image files:", len(image_files))
print("Example image paths:", image_files[:5])


Dataset shape: (210122, 26)
                 id   latitude  longitude  \
0   547473234108938 -16.336027  45.628280   
1   826109781317024  50.855687  56.147997   
2  1006398440000844  37.956651  14.954485   
3  2943891539215481  12.373333  -8.909906   
4   122945119799579   7.510295  99.061884   

                                  thumb_original_url country  \
0  https://scontent-cdg4-3.xx.fbcdn.net/m1/v/t6/A...      MG   
1  https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...      KZ   
2  https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...      IT   
3  https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...      ML   
4  https://scontent-cdg4-3.xx.fbcdn.net/m1/v/t6/A...      TH   

                 sequence    captured_at  lon_bin  lat_bin      cell  ...  \
0  Z2sJMyDVAtHBu7UFGvnYXg  1660909818797       61       28  (61, 28)  ...   
1  6cTpNBSYQH6SeDsfz_0mEQ  1548919530404       64       78  (64, 78)  ...   
2  xhcEpYg3dKnzts5ejIiSBJ  1658912403000       53       68  (53, 68)  ...   
3  j6nwj

In [23]:
from pathlib import Path


image_dir = Path("datasets/OpenWorld/extracted_test")

# Find images inside extracted_test and all its subfolders
image_lookup = {
    p.stem: str(p)
    for p in image_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg"}
}

# Match image IDs to the CSV
df["id"] = df["id"].astype(str).str.strip()
df["image_path"] = df["id"].map(image_lookup)

print("Unique image IDs:", len(image_lookup))
print("Unique CSV IDs:", df["id"].nunique())
print("Matching IDs:", df["image_path"].notna().sum())
print("CSV IDs missing images:", df["image_path"].isna().sum())

print("\nExample matches:")
print(df[["id", "image_path", "country"]].head())

Unique image IDs: 210122
Unique CSV IDs: 210122
Matching IDs: 210122
CSV IDs missing images: 0

Example matches:
                 id                                         image_path country
0   547473234108938  datasets\OpenWorld\extracted_test\03\547473234...      MG
1   826109781317024  datasets\OpenWorld\extracted_test\01\826109781...      KZ
2  1006398440000844  datasets\OpenWorld\extracted_test\00\100639844...      IT
3  2943891539215481  datasets\OpenWorld\extracted_test\00\294389153...      ML
4   122945119799579  datasets\OpenWorld\extracted_test\00\122945119...      TH


In [ ]:
baseline_country = "US" #this was the highest occuring country in dataset

data = df["country"]
y_pred = [baseline_country] * len(data)

accuracy = (data == y_pred).mean()

print("Baseline country:", baseline_country)
print("Baseline accuracy:", accuracy)
print("Number of images:", len(data))
print("Number of correct guesses:", len(data) * accuracy)
print("Number of countries:", data.nunique())


Baseline country: US
Baseline accuracy: 0.11362446578654306
Number of images: 210122
Number of correct guesses: 23875.0
Number of countries: 219


In [24]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

resnet = models.resnet50(pretrained=True)
resnet.eval()  # Set the model to evaluation mode

# Define the image transformation
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_image(image_path):
    image = Image.open(image_path).convert("RGB")
    return transform(image).unsqueeze(0)  # Add batch dimension

def predict_image(image_path):
    image_tensor = load_image(image_path)
    with torch.no_grad():
        outputs = resnet(image_tensor)
    _, predicted = outputs.max(1)
    return predicted.item()

# Example usage:
# predicted_class = predict_image("path/to/your/image.jpg")
# print("Predicted class:", predicted_class)




c:\Users\parkerij\CSSE 416\csse-416-geoguessr-project\.venv\Lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\parkerij\CSSE 416\csse-416-geoguessr-project\.venv\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
